# Figure 5 — Pseudobulk Benchmark


## Setup


In [ ]:
library(ggplot2)
library(patchwork)
library(cowplot)
library(dplyr)
library(tidyr)
library(scales)
library(ggrepel)
library(stringr)
library(yaml)
library(here)
library(grid)
library(data.table)
library(ggpubr)

## Styling and helpers


In [ ]:
AXIS_TITLE_SIZE   <- 7
AXIS_LABEL_SIZE   <- 5.0
PLOT_TITLE_SIZE   <- 5.8
LEGEND_TEXT_SIZE  <- 5.0
LEGEND_TITLE_SIZE <- 6.0

theme_pub <- function(base_size = 7, base_line_size = 0.35) {
    theme_classic(base_size = base_size, base_family = 'Helvetica') %+replace%
    theme(
        plot.title        = element_text(size = PLOT_TITLE_SIZE, face = 'plain', hjust = 0.5,
                                         margin = margin(b = 2)),
        axis.line         = element_line(linewidth = base_line_size),
        axis.ticks        = element_line(linewidth = base_line_size),
        axis.text         = element_text(size = AXIS_LABEL_SIZE),
        axis.title        = element_text(size = AXIS_TITLE_SIZE, face = 'plain',
                                         margin = margin(t = 0, r = 0, b = 0, l = 0)),
        legend.text       = element_text(size = LEGEND_TEXT_SIZE),
        legend.title      = element_text(size = LEGEND_TITLE_SIZE, face = 'plain'),
        legend.key.size   = unit(3, 'mm'),
        legend.background = element_blank(),
        legend.key        = element_blank(),
        legend.position   = 'right',
        panel.grid        = element_blank(),
        strip.text        = element_text(size = PLOT_TITLE_SIZE, face = 'plain'),
        strip.background  = element_blank(),
        plot.margin       = unit(c(1, 2, 1, 1), 'mm')
    )
}

green_colors <- c('#d9e6e2', '#4bc17c', '#007a33')
AUC_PATH     <- 0.60

cfg               <- yaml::read_yaml(here('config.yaml'))
METHOD_COLORS_RAW <- setNames(unlist(cfg$MODEL_COLORS), names(unlist(cfg$MODEL_COLORS)))

ct_labels_df <- read.csv(here('data', 'pseudobulk', 'cell_type_labels.csv'),
                         stringsAsFactors = FALSE)
CT_LABELS    <- setNames(ct_labels_df$label, ct_labels_df$cell_type)
ct_label_fn  <- function(x) ifelse(x %in% names(CT_LABELS), CT_LABELS[x], x)

FIG5_DIR <- here('output', '99_panels', 'fig5', 'fig5')
dir.create(FIG5_DIR, recursive = TRUE, showWarnings = FALSE)

In [ ]:
clean_pw <- function(x) {
    x <- gsub('^(BP_|C2CP_|CellMarker_)', '', x)
    x <- gsub(' \\(GO:[0-9]+\\)', '', x)
    x <- gsub('_', ' ', x)
    stringr::str_wrap(stringr::str_trunc(tools::toTitleCase(tolower(x)), 55), width = 35)
}

clean_c8 <- function(x) {
    sapply(x, function(s) {
        parts <- strsplit(s, '_')[[1]]
        body  <- if (length(parts) > 3) paste(parts[-(1:2)], collapse = ' ')
                 else gsub('_', ' ', s)
        stringr::str_wrap(stringr::str_trunc(tools::toTitleCase(tolower(body)), 55), 35)
    }, USE.NAMES = FALSE)
}

clean_c7 <- function(x) {
    x <- sub('^GSE[0-9]+_', '', x)
    x <- gsub('_VS_', ' vs ', x)
    x <- sub('_(UP|DN)$', '', x)
    x <- gsub('_', ' ', x)
    stringr::str_wrap(stringr::str_trunc(tools::toTitleCase(tolower(x)), 55), 35)
}

clean_allen <- function(x) {
    x <- sub(' up$', '', x)
    x <- sub('^(Human|Mouse) ', '', x)
    stringr::str_wrap(stringr::str_trunc(x, 55), 35)
}

clean_ora_term <- function(term, db) {
    dplyr::case_when(
        db == 'allen' ~ clean_allen(term),
        db == 'c7'    ~ clean_c7(term),
        TRUE          ~ clean_c8(term)
    )
}

# Bubble plot theme for ~90mm-wide panels
bubble_theme_bio <- theme(
    panel.background   = element_rect(fill = 'white', color = NA),
    plot.background    = element_rect(fill = 'white', color = NA),
    panel.grid.major.x = element_line(linetype = 'dotted', linewidth = 0.35, color = '#cccccc'),
    panel.grid.major.y = element_blank(),
    panel.grid.minor   = element_blank(),
    axis.line.x        = element_line(color = '#444444', linewidth = 0.35),
    axis.line.y        = element_blank(),
    axis.ticks.y       = element_blank(),
    axis.text.y        = element_text(size = 5.5, hjust = 1, lineheight = 0.85,
                                      family = 'Helvetica'),
    axis.text.x        = element_text(size = 5.5, family = 'Helvetica'),
    axis.title.x       = element_text(size = 6, margin = margin(t = 3),
                                      family = 'Helvetica'),
    legend.position    = 'none',
    plot.margin        = margin(2, 8, 3, 5)
)

## Row 1: Benchmark


In [ ]:
long <- data.table::fread(file.path(FIG5_DIR, 'benchmark_long.csv'))
long_box <- long[truth == 'v0' & !is.na(cor)]

# Method order by median correlation (descending)
med_order <- long_box[, .(med = median(cor, na.rm = TRUE)), by = method]
data.table::setorder(med_order, -med)
method_order <- as.character(med_order$method)
long_box[, method := factor(method, levels = method_order)]

# Map config colors to the methods present in the data
METHOD_COLORS <- METHOD_COLORS_RAW[intersect(names(METHOD_COLORS_RAW), method_order)]
missing_m <- setdiff(method_order, names(METHOD_COLORS))
if (length(missing_m) > 0) {
    METHOD_COLORS <- c(METHOD_COLORS, setNames(rep('grey60', length(missing_m)), missing_m))
}
METHOD_COLORS <- METHOD_COLORS[method_order]

comparisons <- lapply(setdiff(method_order, 'CLAMPfull')[1:3],
                      function(m) c('CLAMPfull', m))

plot_A <- ggplot(long_box, aes(method, cor, fill = method)) +
    geom_boxplot(outlier.shape = NA, width = 0.5, linewidth = 0.28) +
    geom_jitter(aes(color = method), width = 0.12, size = 1.0, alpha = 0.65,
                show.legend = FALSE) +
    scale_fill_manual(values  = METHOD_COLORS) +
    scale_color_manual(values = METHOD_COLORS) +
    scale_y_continuous(breaks = seq(0, 1, 0.25), limits = c(0, NA)) +
    ggpubr::stat_compare_means(comparisons = comparisons, method = 'wilcox.test',
                               label = 'p.signif', tip.length = 0.01,
                               step.increase = 0.06, size = 2.0) +
    stat_summary(fun = median, geom = 'text',
                 aes(label = sprintf('%.3f', after_stat(y))),
                 vjust = -0.5, size = 1.8, color = 'black') +
    theme_pub() +
    theme(legend.position = 'none',
          axis.text.x = element_text(angle = 35, hjust = 1, size = AXIS_LABEL_SIZE)) +
    labs(x = NULL, y = 'Max Pearson r per cell type')

options(repr.plot.width = 10, repr.plot.height = 4)
print(plot_A)

## Row 2: Hard-to-distinguish cell type pairs


In [ ]:
pair_pathways    <- read.csv(file.path(FIG5_DIR, 'pair_pathways.csv'),      stringsAsFactors = FALSE)
pair_ora         <- read.csv(file.path(FIG5_DIR, 'pair_ora.csv'),           stringsAsFactors = FALSE)
pair_assignments <- read.csv(file.path(FIG5_DIR, 'pair_lv_assignments.csv'), stringsAsFactors = FALSE)

db_display <- c(
    'GO-BP' = 'GO Biological Process',
    'c7'    = 'C7 Immunological Signatures',
    'c8'    = 'C8 Cell Types',
    'allen' = 'Allen Brain Atlas'
)

PAIRS <- list(
    list(ds = 'PBMC_1k1k',
         lvs = c(CD14_Mono = 'LV2',  CD16_Mono = 'LV20'),
         pref_db = 'c7',
         panel_label = 'CD14+ vs CD16+ Monocytes  [PBMC_1k1k]'),
    list(ds = 'Brain_Xiong2023',
         lvs = c(Opc = 'LV11', Oli = 'LV14'),
         pref_db = 'allen',
         panel_label = 'OPC vs Oligodendrocyte  [Brain_Xiong2023]'),
    list(ds = 'Brain_Mathys2023',
         lvs = c(Ast = 'LV17', Mic = 'LV6'),
         pref_db = 'allen',
         panel_label = 'Astrocyte vs Microglia  [Brain_Mathys2023]')
)

# Single-section lollipop plot (no facets)
make_bubble_section <- function(df, section_title, v_max, is_bottom = FALSE) {
    df <- df[order(df$neg_log_p), ]
    df$term_clean <- factor(df$term_clean, levels = unique(df$term_clean))

    ggplot(df, aes(x = neg_log_p, y = term_clean)) +
        geom_segment(aes(x = 0, xend = neg_log_p, yend = term_clean),
                     color = '#cccccc', linewidth = 0.4, linetype = 'dotted') +
        geom_point(aes(color = neg_log_p, size = bubble_size)) +
        scale_color_gradientn(colors = green_colors, limits = c(0, v_max)) +
        scale_size_identity(guide = 'none') +
        scale_x_continuous(limits = c(0, v_max + 0.25),
                           breaks = c(0, round(v_max / 2), v_max)) +
        labs(x = if (is_bottom) expression(-log[10] ~ 'FDR') else NULL,
             y = NULL, title = section_title) +
        bubble_theme_bio +
        theme(
            plot.title   = element_text(size = 5, face = 'bold', hjust = 0,
                                         margin = margin(b = 1), family = 'Helvetica',
                                         color = '#444444'),
            axis.title.x = element_text(size = 5, margin = margin(t = 2),
                                         family = 'Helvetica'),
            axis.text.x  = if (is_bottom) element_text(size = 4.5) else element_blank(),
            axis.ticks.x = if (is_bottom) element_line(linewidth = 0.3) else element_blank(),
            axis.line.x  = if (is_bottom) element_line(color = '#333333', linewidth = 0.35)
                           else element_blank(),
            plot.margin  = margin(1, 6, if (is_bottom) 2 else 0, 22)
        )
}

make_cell_type_bubble <- function(ds, lv, pref_db) {
    pa_row  <- pair_assignments[pair_assignments$dataset == ds & pair_assignments$LV == lv, ]
    ct      <- if (nrow(pa_row) > 0) pa_row$cell_type[1] else lv
    ct_str  <- stringr::str_trunc(ct_label_fn(ct), 22)
    cor_val <- if (nrow(pa_row) > 0) max(pa_row$cor, na.rm = TRUE) else NA_real_

    # GO-BP pathways (top 3 by lowest FDR)
    clamp_sub <- pair_pathways[pair_pathways$dataset == ds & pair_pathways$LV == lv, ]
    go_df <- NULL
    if (nrow(clamp_sub) > 0) {
        clamp_sub <- clamp_sub[order(clamp_sub$FDR), ][seq_len(min(3, nrow(clamp_sub))), ]
        go_df <- data.frame(
            term_clean  = clean_pw(clamp_sub$pathway),
            neg_log_p   = pmin(-log10(clamp_sub$FDR + 1e-300), 4),
            bubble_size = 1 + pmax(0, (clamp_sub$AUC - AUC_PATH) / (1 - AUC_PATH)) * 4,
            stringsAsFactors = FALSE
        )
    }

    # Preferred ORA db only (top 4 by lowest padj)
    ora_sub <- pair_ora[pair_ora$dataset == ds & pair_ora$LV == lv & pair_ora$db == pref_db, ]
    ora_df <- NULL
    if (nrow(ora_sub) > 0) {
        ora_sub <- ora_sub[order(ora_sub$padj), ][seq_len(min(4, nrow(ora_sub))), ]
        ora_df <- data.frame(
            term_clean  = clean_ora_term(ora_sub$term, ora_sub$db),
            neg_log_p   = pmin(-log10(ora_sub$padj + 1e-300), 4),
            bubble_size = 1 + pmin(ora_sub$fold_enrichment / 30, 1) * 4,
            stringsAsFactors = FALSE
        )
    }

    has_go  <- !is.null(go_df) && nrow(go_df) > 0
    has_ora <- !is.null(ora_df) && nrow(ora_df) > 0

    if (!has_go && !has_ora) {
        return(ggplot() +
            annotate('text', x = 0.5, y = 0.5,
                     label = paste0(ct_str, '\nno enrichment'),
                     hjust = 0.5, vjust = 0.5, size = 2, color = 'grey50',
                     family = 'Helvetica') +
            theme_void(base_family = 'Helvetica'))
    }

    all_vals <- c(if (has_go) go_df$neg_log_p else NULL,
                  if (has_ora) ora_df$neg_log_p else NULL)
    v_max    <- max(2, ceiling(max(all_vals, na.rm = TRUE)))

    sections <- list()
    heights  <- c()

    if (has_go) {
        sections <- c(sections, list(make_bubble_section(
            go_df, 'GO Biological Process', v_max, is_bottom = !has_ora)))
        heights  <- c(heights, nrow(go_df) + 0.8)
    }
    if (has_ora) {
        sections <- c(sections, list(make_bubble_section(
            ora_df, unname(db_display[pref_db]), v_max, is_bottom = TRUE)))
        heights  <- c(heights, nrow(ora_df) + 0.8)
    }

    title_grob <- cowplot::ggdraw() +
        cowplot::draw_label(
            sprintf('%s\n%s  (r=%.2f)', lv, ct_str, cor_val),
            fontface = 'bold', fontfamily = 'Helvetica', size = 5.5,
            hjust = 0.5, vjust = 0.5, lineheight = 0.9
        )

    stacked <- patchwork::wrap_plots(sections, ncol = 1, heights = heights)
    cowplot::plot_grid(title_grob, stacked, ncol = 1, rel_heights = c(0.12, 1))
}

make_pair_plot <- function(ep) {
    panels <- lapply(seq_along(ep$lvs), function(i) {
        lv <- ep$lvs[i]
        make_cell_type_bubble(ep$ds, lv, ep$pref_db)
    })
    # Left-aligned header to avoid right-edge clipping on long dataset names
    header <- cowplot::ggdraw() +
        cowplot::draw_label(ep$panel_label,
                            fontface = 'bold', fontfamily = 'Helvetica',
                            size = 5.8, hjust = 0, x = 0.03, vjust = 0.5)
    body   <- cowplot::plot_grid(plotlist = panels, ncol = 2, align = 'h')
    cowplot::plot_grid(header, body, ncol = 1, rel_heights = c(0.09, 1))
}

pair_plots <- lapply(PAIRS, make_pair_plot)
names(pair_plots) <- c('B', 'C', 'D')

options(repr.plot.width = 20, repr.plot.height = 10)
for (nm in names(pair_plots)) {
    cat('Pair', nm, '\n')
    print(pair_plots[[nm]])
}

## Assembly and export


In [ ]:
FIG_W <- 183
FIG_H <- 300

# ── Legend grob ──────────────────────────────────────────────────────────────
make_legend_grob <- function(v_max = 4) {
    n      <- 50
    col_df <- data.frame(
        x   = seq(0, v_max, length.out = n),
        y   = 5.0,
        col = seq(0, v_max, length.out = n)
    )

    auc_ref <- c(0.65, 0.80, 1.00)
    go_df   <- data.frame(
        x  = c(0.7, 2.0, 3.3),
        y  = 3.0,
        sz = 1 + pmax(0, (auc_ref - AUC_PATH) / (1 - AUC_PATH)) * 4,
        lb = c('0.65', '0.80', '1.0')
    )

    fe_ref  <- c(10, 20, 30)
    ora_df  <- data.frame(
        x  = c(0.7, 2.0, 3.3),
        y  = 0.8,
        sz = 1 + pmin(fe_ref / 30, 1) * 4,
        lb = c('10x', '20x', '>30x')
    )

    lbl_df <- data.frame(
        x     = c(-0.1, -0.1, -0.1),
        y     = c(5.75,  3.70,  1.50),
        label = c('paste(-log[10], "(FDR)")',
                  '"Dot size: GSEA AUC"',
                  '"Dot size: fold enrichment"')
    )

    tick_df <- data.frame(
        x   = c(0, v_max / 2, v_max),
        y   = 4.30,
        lbl = c('0', as.character(v_max / 2), as.character(v_max))
    )

    ggplot() +
        geom_tile(data = col_df, aes(x = x, y = y, fill = col), height = 0.55) +
        scale_fill_gradientn(colors = green_colors, limits = c(0, v_max),
                             guide = 'none') +
        geom_point(data = go_df,  aes(x = x, y = y, size = sz),
                   color = '#007a33', alpha = 0.85) +
        geom_text(data  = go_df,  aes(x = x, y = y - 0.65, label = lb),
                  size = 1.6, family = 'Helvetica', color = '#444444',
                  hjust = 0.5) +
        geom_point(data = ora_df, aes(x = x, y = y, size = sz),
                   color = '#007a33', alpha = 0.85) +
        geom_text(data  = ora_df, aes(x = x, y = y - 0.65, label = lb),
                  size = 1.6, family = 'Helvetica', color = '#444444',
                  hjust = 0.5) +
        geom_text(data = lbl_df, aes(x = x, y = y, label = label),
                  parse = TRUE, size = 1.8, hjust = 0,
                  family = 'Helvetica', color = '#222222') +
        geom_text(data = tick_df, aes(x = x, y = y, label = lbl),
                  size = 1.5, hjust = 0.5, family = 'Helvetica',
                  color = '#444444') +
        scale_size_identity() +
        coord_cartesian(clip = 'off',
                        xlim = c(-0.3, v_max + 0.5),
                        ylim = c(-0.3, 6.2)) +
        theme_void(base_family = 'Helvetica') +
        theme(plot.background = element_rect(fill = 'white', color = NA),
              plot.margin     = margin(10, 2, 2, 4))
}

legend_grob <- make_legend_grob()

# ── Assembly ─────────────────────────────────────────────────────────────────
options(repr.plot.width = 7.2, repr.plot.height = 11.8)

# Row 1: benchmark (left 78%) + legend (right 22%)
row1 <- cowplot::plot_grid(
    plot_A, legend_grob,
    ncol = 2, rel_widths = c(0.78, 0.22),
    labels = c('A', ''), label_size = 8,
    label_fontface = 'bold', label_fontfamily = 'Helvetica'
)

# Row 2: 3 pair rows stacked vertically; each pair = 2 cell types side-by-side
row2 <- cowplot::plot_grid(
    pair_plots[['B']], pair_plots[['C']], pair_plots[['D']],
    ncol = 1, rel_heights = c(1, 1, 1),
    labels = c('B', 'C', 'D'),
    label_size = 8, label_fontface = 'bold', label_fontfamily = 'Helvetica'
)

fig5 <- cowplot::plot_grid(
    row1, ggdraw(), row2,
    ncol = 1,
    rel_heights = c(0.29, 0.015, 0.695)
)

ggsave(file.path(FIG5_DIR, 'fig5.pdf'), fig5,
       width = FIG_W, height = FIG_H, units = 'mm',
       device = cairo_pdf, bg = 'white')
ggsave(file.path(FIG5_DIR, 'fig5.png'), fig5,
       width = FIG_W, height = FIG_H, units = 'mm',
       dpi = 300, bg = 'white')
cat('fig5 exported to', FIG5_DIR, '\n')
fig5